# Unauthorized Access & Behavioral Biometrics

### Cybersecurity Threat Detection — Banking-AI

Traditional passwords can be stolen. Behavioral biometrics adds a layer of security by analyzing *how* a user interacts with their device (typing speed, rhythm, mouse usage). If the behavior changes, it might be an intruder.

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of cybersecurity and threat detection terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the cybersecurity and threat detection problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** DDoS detection, phishing classification, behavioural biometrics, and endpoint security in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Even if an attacker has your password, they likely don't have your exact typing rhythm. This acts as a "silent" multi-factor authentication.

**Input data used:** Typing speed (WPM), Average key-press duration, Error rate (backspaces), and mouse movement fluidity.

**What we predict:** Identity Verification (Genuine vs Intruder).

**ML method used:** Support Vector Machine (SVM) - excellent for finding clear boundaries between two classes.

**Learning goal:** Learn how to use "soft biometrics" for security.

**Primary stakeholders:** security operations analysts, incident responders, CISOs, and fraud prevention teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We'll simulate behavior for 1,000 sessions.

In [ ]:
n_sessions = 1000

# Genuine User: Consistent typing speed, few errors, predictable mouse movement
gen_typing_speed = RNG.normal(70, 5, n_sessions // 2) # 70 WPM
gen_error_rate = RNG.normal(2, 0.5, n_sessions // 2) # 2% error
gen_hold_time = RNG.normal(100, 10, n_sessions // 2) # 100ms key hold

# Intruder: Faster or slower typing, more mistakes (unfamiliar with UI), jittery mouse
int_typing_speed = RNG.normal(40, 15, n_sessions // 2) # Slower/unsteady
int_error_rate = RNG.normal(8, 2, n_sessions // 2) # 8% error
int_hold_time = RNG.normal(150, 30, n_sessions // 2) # 150ms key hold

df_gen = pd.DataFrame({
    'typing_speed': gen_typing_speed,
    'error_rate': gen_error_rate,
    'hold_time_ms': gen_hold_time,
    'label': 1 # Genuine
})

df_int = pd.DataFrame({
    'typing_speed': int_typing_speed,
    'error_rate': int_error_rate,
    'hold_time_ms': int_hold_time,
    'label': 0 # Intruder
})

df = pd.concat([df_gen, df_int]).sample(frac=1).reset_index(drop=True)
print(f"Dataset created with {len(df)} behavioral sessions.")

## Step 4 — Exploratory Data Analysis

EDA

Let's see if typing speed and error rate can help us distinguish the user.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='typing_speed', y='error_rate', hue='label', data=df, alpha=0.7)
plt.title('Behavioral Profile: Typing Speed vs Error Rate')
plt.legend(title='Identity', labels=['Intruder', 'Genuine'])
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "Behavioral Profile: Typing Speed vs Error Rate". The chart highlights spatial separation or clustering patterns in the data.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = SVC(kernel='rbf', probability=True)
model.fit(X_train_scaled, y_train)

print("Identity Verification model trained.")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train_scaled, y_train)
baseline_pred = baseline_clf.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=['Intruder', 'Genuine']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Intruder', 'Genuine'], yticklabels=['Intruder', 'Genuine'])
plt.title('Verification Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "Verification Confusion Matrix" with Predicted on the x-axis and Actual on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

The model successfully separates the genuine user from the intruder. By adjusting the "decision threshold," a bank can choose to be more or less strict. For a sensitive action (like a $10,000 transfer), the bank might require a very high behavioral match score.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end cybersecurity and threat detection workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Threat detection models must balance security effectiveness with employee and customer privacy.
- False positives in security alerts cause alert fatigue and reduce team effectiveness.
- Behavioural biometrics raise consent and surveillance concerns that require clear policies.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in cybersecurity and threat detection?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.